### Superseeded - predates leakage fix described in README, see 04 for final version. Kept as a audit-trail
This notebook handles mutli-cohort validation using GSE16879 (Batch 1) and GSE23597 (Batch 2)


In [1]:
import pandas as pd
import numpy as np
import sklearn
import os
import urllib.request
import gzip

#function to download data using GSEID
def download_geo_matrix(gse_id):
    short_gse_id = gse_id[:5]
    data_url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{short_gse_id}nnn/{gse_id}/matrix/{gse_id}_series_matrix.txt.gz" # define URL 
    dest_path = os.path.join("data", f"{gse_id}_series_matrix.txt.gz") #define local path to save the file
    os.makedirs("data", exist_ok = True)
    if not os.path.exists(dest_path):
        print("Downloading dataset from NCBI GEO... This might take a minute.")
        urllib.request.urlretrieve(data_url, dest_path)
        print(f"Download complete! File saved to: {dest_path}")
    else:
        print("Dataset already exists locally in the 'data' folder.")
        
download_geo_matrix("GSE23597")
     
                                   




Dataset already exists locally in the 'data' folder.


In [ ]:
#reading first 100 lines of the file
file_path = os.path.join("data", "GSE23597_series_matrix.txt.gz")
with gzip.open(file_path, "rt", encoding = 'utf8', errors = 'ignore') as f:
    for i in range(100):
        line = f.readline().strip()
        print(f"Line {i+1}: {line[:120]}")

In [ ]:
#parsing data
with gzip.open(file_path, 'rt', encoding = 'utf8', errors = 'ignore') as f:
    for line in f:
        clean_line = line.strip()
        if len(clean_line) == 0:
            continue
        elif clean_line.startswith("!") == False: #if line is empty or does not start with "!"
            break
        if clean_line.startswith("!Sample"):
            print(line[:50])
        
        



In [ ]:
metadata_dict = {}
#parsing data
with gzip.open(file_path, 'rt', encoding = 'utf8', errors = 'ignore') as f:
    for line in f:
        clean_line = line.strip()
        if len(clean_line) == 0:
            continue
        elif clean_line.startswith("!") == False: #if line is empty or does not start with "!"
            break
        if clean_line.startswith("!Sample"):
            parts = clean_line.split("\t")
            base_key = parts[0].replace("!Sample_", "")
            values = [v.replace('"', "").strip() for v in parts[1:]] 
            final_key = base_key
            counter = 1
            while final_key in  metadata_dict:
                final_key = f"{base_key}_{counter}"
                counter += 1
            metadata_dict[final_key] = values
df_clinical_batch2 = pd.DataFrame(metadata_dict)
list(df_clinical_batch2.columns)


In [ ]:
df_clinical_batch2.filter(like='characteristics').head(5) # selectively output "characteristics" columns

In [ ]:
#rename columns
df_clinical_batch2 = df_clinical_batch2.rename(columns = {'characteristics_ch1_2' : 'Time'})
df_clinical_batch2 = df_clinical_batch2.rename(columns = {'characteristics_ch1_4' : 'Primary Response'})

#new dataframe only containing rows where Time = W0 
df_baseline = df_clinical_batch2[df_clinical_batch2['Time'] == "time: W0"].copy()

#cleaning cells
df_baseline['Time'] = df_baseline['Time'].str.replace("time: ", "")
df_baseline['Primary Response'] = df_baseline['Primary Response'].str.replace("wk8 response: ", "")
df_baseline[['Time', 'Primary Response']].head()

In [ ]:
df_genes_batch2 = pd.read_csv(file_path, 
                              sep = '\t',          #columns separated by tabs 
                              comment = "!",       # ignores rows starting with "!"
                              index_col = 0,       #column 0 used as row labels
                              compression = 'gzip')
df_genes_batch2.head()

In [ ]:
#transpose table - rows must be independent samples, not genes
df_genes_batch2 = df_genes_batch2.T


In [ ]:
df_baseline = df_baseline.set_index('geo_accession') #align row names
df_final_batch2 = pd.merge(df_baseline,df_genes_batch2, left_index= True, right_index=True, how = 'inner') #merges dataframes, only including where data is available in both dfs
df_final_batch2.shape

In [ ]:
df_final_batch2.head()

In [ ]:
y = df_final_batch2['Primary Response']
X = df_final_batch2.drop(columns = df_baseline.columns) #drops columns from df_baseline

print(f"X shape:{X.shape}")

print(f"y shape: {y.shape}")

In [ ]:
import joblib
from sklearn.metrics import accuracy_score

#importing model created in 02_final_pipeline.ipynb
loaded_model = joblib.load("infliximab_lasso_model.pkl")

#model predicts response using genes
predictions = loaded_model.predict(X)

#predictions compared against answers in y to provide accuracy
accuracy = accuracy_score(predictions, y)

print(f"The model achieved {accuracy *100: .2f}%")
